# Nexus-Steg

**Robust Semantic-Texture Hybrid Steganography**

This notebook clones the repository, downloads datasets, trains the model, evaluates it against real-world attacks, and visualizes the architecture — all on Google Colab with GPU acceleration.

## 1. Setup

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > T4 GPU"

In [ ]:
!git clone https://github.com/Henildiyora/Nexus-Steg.git
%cd Nexus-Steg

In [ ]:
!pip install -q torch torchvision tqdm pillow tifffile torchviz graphviz
!apt-get install -qq graphviz > /dev/null 2>&1

## 2. Download Datasets

Downloads MS-COCO val2017 (covers) and SpaceNet 2 satellite imagery (secrets) for all 4 cities.

In [ ]:
!bash scripts/download_data.sh

In [ ]:
import os

def count_images(path):
    if not os.path.isdir(path):
        return 0
    exts = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
    return sum(1 for root, _, files in os.walk(path) for f in files if f.lower().endswith(exts))

print(f"Cover images:        {count_images('datasets/cover')}")
print(f"Secret train images: {count_images('datasets/secret/train')}")
print(f"Secret test images:  {count_images('datasets/secret/test')}")

## 3. Health Check

Verify the architecture is wired correctly before training.

In [ ]:
!python check_health.py

## 4. Train

Runs 100-epoch phase-based training:
- **Phase 1** (0-29): Pure hiding/recovery
- **Phase 2** (30-59): Noise robustness + mild adversarial
- **Phase 3** (60-99): Full adversarial training

Checkpoints are saved every 10 epochs to `checkpoints/`.

In [ ]:
!python main.py

## 5. Save Checkpoints to Google Drive (Optional)

Mount Drive and copy checkpoints so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_dest = '/content/drive/MyDrive/Nexus-Steg'
os.makedirs(drive_dest, exist_ok=True)

!cp -v checkpoints/*.pth "{drive_dest}/"
!cp -rv results/ "{drive_dest}/results/"
print(f"\nSaved to {drive_dest}")